# DBT Lineage Diagnostics

Notebook pour diagnostiquer les coupures de data et les prix plats a travers toutes les couches du pipeline:
- `staging`
- `intermediate`
- `marts`

Objectifs:
- visualiser toutes les tables materialisees par dbt
- comparer la lineage du prix `stg_swaps -> int_daily_close -> mart_tvl_daily`
- detecter les streaks de prix constants
- localiser les trous de dates par couche


## Prerequis

Si besoin, decommenter et executer:

```python
%pip install pandas plotly clickhouse-connect jupyterlab
```


In [ ]:
import os
from textwrap import dedent

import clickhouse_connect
from IPython.display import display

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "pandas is required. Run: %pip install pandas plotly clickhouse-connect jupyterlab"
    ) from exc

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "plotly is required. Run: %pip install plotly"
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


In [ ]:
# Defaults alignes avec dbt/profiles.yml, avec surcharge possible par variables d'environnement.
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT", "8124"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "1234")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "analytics")

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

print(f"Connected to ClickHouse database={CLICKHOUSE_DATABASE} on {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}")


Connected to ClickHouse database=analytics on localhost:8123


In [ ]:
TIME_COLUMNS = ["date", "block_time", "start_time", "end_time", "deployment_ts"]


def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    for column in TIME_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column])
    return df


def query_df(sql: str, parameters: dict | None = None) -> pd.DataFrame:
    df = client.query_df(sql, parameters=parameters or {})
    df = normalize_df(df)
    print(f"{len(df):,} rows")
    return df


def build_filters(
    pool: str | None = None,
    date_from: str | None = None,
    date_to: str | None = None,
    pool_col: str | None = "pool",
    time_col: str | None = "date",
    time_type: str = "date",
):
    clauses = []
    params = {}

    if pool and pool_col:
        clauses.append(f"{pool_col} = %(pool)s")
        params["pool"] = pool

    if date_from and time_col:
        caster = "toDate" if time_type == "date" else "toDateTime"
        clauses.append(f"{time_col} >= {caster}(%(date_from)s)")
        params["date_from"] = date_from

    if date_to and time_col:
        if time_type == "date":
            clauses.append(f"{time_col} <= toDate(%(date_to)s)")
        else:
            clauses.append(f"{time_col} < addDays(toDate(%(date_to)s), 1)")
        params["date_to"] = date_to

    where_sql = ""
    if clauses:
        where_sql = "WHERE " + " AND ".join(clauses)
    return where_sql, params


def plot_timeseries(df: pd.DataFrame, title: str, bar_cols: list[str] | None = None, line_cols: list[str] | None = None):
    if df.empty:
        print(f"No data for: {title}")
        return

    bar_cols = bar_cols or []
    line_cols = line_cols or []
    fig = go.Figure()

    for column in bar_cols:
        if column in df.columns:
            fig.add_trace(go.Bar(x=df["date"], y=df[column], name=column))

    for column in line_cols:
        if column in df.columns:
            fig.add_trace(go.Scatter(x=df["date"], y=df[column], mode="lines+markers", name=column))

    fig.update_layout(title=title, template="plotly_white", hovermode="x unified")
    fig.show()


def constant_streaks(df: pd.DataFrame, value_col: str, min_days: int = 3) -> pd.DataFrame:
    if df.empty or value_col not in df.columns:
        return pd.DataFrame(columns=["start_date", "end_date", "days", "value"])

    work = df[["date", value_col]].dropna().copy()
    if work.empty:
        return pd.DataFrame(columns=["start_date", "end_date", "days", "value"])

    work["date"] = pd.to_datetime(work["date"]).dt.normalize()
    work = work.sort_values("date")
    work["same_value"] = work[value_col].eq(work[value_col].shift())
    work["next_day"] = work["date"].diff().dt.days.eq(1)
    work["new_group"] = ~(work["same_value"] & work["next_day"])
    work["group_id"] = work["new_group"].cumsum()

    streaks = (
        work.groupby("group_id", as_index=False)
        .agg(
            start_date=("date", "min"),
            end_date=("date", "max"),
            days=("date", "count"),
            value=(value_col, "first"),
        )
    )
    return streaks.loc[streaks["days"] >= min_days].sort_values(["days", "start_date"], ascending=[False, True])


def gap_summary(df: pd.DataFrame) -> tuple[int, int]:
    if df.empty or "date" not in df.columns:
        return 0, 0

    dates = pd.Series(pd.to_datetime(df["date"]).dt.normalize().dropna().unique()).sort_values()
    if dates.empty:
        return 0, 0

    missing_days = 0
    longest_gap_days = 0

    if len(dates) > 1:
        gaps = dates.diff().dt.days.fillna(1).astype(int) - 1
        gaps = gaps.clip(lower=0)
        missing_days = int(gaps.sum())
        longest_gap_days = int(gaps.max())

    return missing_days, longest_gap_days


def preview(table_name: str, limit: int = 20) -> pd.DataFrame:
    sql = f"SELECT * FROM {table_name} LIMIT {int(limit)}"
    return query_df(sql)


In [ ]:
TIME_SERIES_SPECS = [
    {
        "table": "stg_swaps",
        "layer": "staging",
        "bucket_expr": "toDate(block_time)",
        "filter_time_col": "block_time",
        "time_type": "datetime",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "sum(abs(toFloat64(amount0))) AS abs_amount0_usdc",
            "argMax(price_token1_in_token0, tuple(block_number, log_index)) AS last_price_token1_in_token0",
        ],
    },
    {
        "table": "stg_mints",
        "layer": "staging",
        "bucket_expr": "toDate(block_time)",
        "filter_time_col": "block_time",
        "time_type": "datetime",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "sum(toFloat64(liquidity_delta)) AS liquidity_delta",
            "sum(abs(toFloat64(amount0))) AS abs_amount0_usdc",
        ],
    },
    {
        "table": "stg_burns",
        "layer": "staging",
        "bucket_expr": "toDate(block_time)",
        "filter_time_col": "block_time",
        "time_type": "datetime",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "sum(toFloat64(liquidity_delta)) AS liquidity_delta",
            "sum(abs(toFloat64(amount0))) AS abs_amount0_usdc",
        ],
    },
    {
        "table": "int_lp_events",
        "layer": "intermediate",
        "bucket_expr": "toDate(block_time)",
        "filter_time_col": "block_time",
        "time_type": "datetime",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "sum(toFloat64(liquidity_delta)) AS net_liquidity_delta",
            "sum(abs(toFloat64(amount0))) AS abs_amount0_usdc",
        ],
    },
    {
        "table": "int_daily_close",
        "layer": "intermediate",
        "bucket_expr": "date",
        "filter_time_col": "date",
        "time_type": "date",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "any(close_tick) AS close_tick",
            "any(close_price) AS close_price",
        ],
    },
    {
        "table": "int_tick_changes",
        "layer": "intermediate",
        "bucket_expr": "toDate(start_time)",
        "filter_time_col": "start_time",
        "time_type": "datetime",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "sum(volume_token0_equiv) AS volume_token0_equiv",
            "sum(swap_count) AS swap_count",
        ],
    },
    {
        "table": "int_range_liquidity_daily",
        "layer": "intermediate",
        "bucket_expr": "date",
        "filter_time_col": "date",
        "time_type": "date",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "countDistinct(tuple(tick_lower, tick_upper)) AS active_ranges",
            "sum(toFloat64(net_liquidity)) AS total_net_liquidity",
        ],
    },
    {
        "table": "int_active_liquidity_daily",
        "layer": "intermediate",
        "bucket_expr": "date",
        "filter_time_col": "date",
        "time_type": "date",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "sum(toFloat64(active_liquidity)) AS active_liquidity",
            "any(close_price) AS close_price",
        ],
    },
    {
        "table": "int_active_tvl_daily",
        "layer": "intermediate",
        "bucket_expr": "date",
        "filter_time_col": "date",
        "time_type": "date",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "sum(active_tvl_token0) AS active_tvl_token0",
            "sum(active_range_count) AS active_range_count",
        ],
    },
    {
        "table": "mart_volume_daily",
        "layer": "mart",
        "bucket_expr": "date",
        "filter_time_col": "date",
        "time_type": "date",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "sum(volume_token0) AS volume_token0",
            "sum(fees_token0_equiv) AS fees_token0_equiv",
            "sum(swap_count) AS swap_count",
        ],
    },
    {
        "table": "mart_tvl_daily",
        "layer": "mart",
        "bucket_expr": "date",
        "filter_time_col": "date",
        "time_type": "date",
        "pool_col": "pool",
        "expected_dense": True,
        "metrics": [
            "count() AS row_count",
            "sum(tvl_token0_equiv) AS tvl_token0_equiv",
            "any(price_token1_in_token0) AS close_price_token1_in_token0",
        ],
    },
    {
        "table": "mart_fee_yield_daily",
        "layer": "mart",
        "bucket_expr": "date",
        "filter_time_col": "date",
        "time_type": "date",
        "pool_col": "pool",
        "expected_dense": True,
        "metrics": [
            "count() AS row_count",
            "sum(tvl_total_token0) AS tvl_total_token0",
            "sum(tvl_active_token0) AS tvl_active_token0",
            "sum(fees_token0_equiv) AS fees_token0_equiv",
        ],
    },
    {
        "table": "mart_tick_daily",
        "layer": "mart",
        "bucket_expr": "date",
        "filter_time_col": "date",
        "time_type": "date",
        "pool_col": "pool",
        "expected_dense": False,
        "metrics": [
            "count() AS row_count",
            "sum(time_seconds) AS time_seconds",
            "sum(volume_token0_equiv) AS volume_token0_equiv",
            "sum(swap_count) AS swap_count",
        ],
    },
]

SNAPSHOT_SPECS = [
    {"table": "stg_pools", "layer": "staging"},
    {"table": "int_current_state", "layer": "intermediate"},
    {"table": "mart_open_positions", "layer": "mart"},
    {"table": "mart_tvl_distribution", "layer": "mart"},
]


def load_daily_series(spec: dict, pool: str | None = None, date_from: str | None = None, date_to: str | None = None) -> pd.DataFrame:
    filters, params = build_filters(
        pool=pool,
        date_from=date_from,
        date_to=date_to,
        pool_col=spec.get("pool_col", "pool"),
        time_col=spec.get("filter_time_col", "date"),
        time_type=spec.get("time_type", "date"),
    )
    metrics_sql = ",\n            ".join(spec["metrics"])
    sql = dedent(
        f"""
        SELECT
            {spec['bucket_expr']} AS date,
            {metrics_sql}
        FROM {spec['table']}
        {filters}
        GROUP BY date
        ORDER BY date
        """
    )
    return query_df(sql, params)


In [ ]:
available_pools = query_df(
    dedent(
        """
        SELECT
            pool,
            min(toDate(block_time)) AS min_date,
            max(toDate(block_time)) AS max_date,
            count() AS swap_rows,
            round(avg(price_token1_in_token0), 2) AS avg_swap_price
        FROM stg_swaps
        GROUP BY pool
        ORDER BY swap_rows DESC, pool
        """
    )
)
available_pools


1 rows


,pool,min_date,max_date,swap_rows,avg_swap_price
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2021-05-05,2026-03-13,10645912,2475.84


In [ ]:
# Ajuste ces parametres pour explorer un autre pool ou une autre fenetre.
POOL = available_pools.iloc[0]["pool"] if not available_pools.empty else None
DATE_FROM = None
DATE_TO = None
CONSTANT_PRICE_MIN_DAYS = 3
PREVIEW_LIMIT = 50

print("POOL:", POOL)
print("DATE_FROM:", DATE_FROM)
print("DATE_TO:", DATE_TO)
print("CONSTANT_PRICE_MIN_DAYS:", CONSTANT_PRICE_MIN_DAYS)


POOL: 0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640
DATE_FROM: None
DATE_TO: None
CONSTANT_PRICE_MIN_DAYS: 3


In [ ]:
inventory_rows = []

for spec in TIME_SERIES_SPECS:
    filters, params = build_filters(
        pool=POOL,
        date_from=DATE_FROM,
        date_to=DATE_TO,
        pool_col=spec.get("pool_col", "pool"),
        time_col=spec.get("filter_time_col", "date"),
        time_type=spec.get("time_type", "date"),
    )
    sql = dedent(
        f"""
        SELECT
            count() AS row_count,
            min(date) AS min_date,
            max(date) AS max_date,
            countDistinct(date) AS distinct_days
        FROM (
            SELECT {spec['bucket_expr']} AS date
            FROM {spec['table']}
            {filters}
        ) t
        """
    )
    stats = query_df(sql, params)
    row = stats.iloc[0].to_dict() if not stats.empty else {}
    inventory_rows.append(
        {
            "table": spec["table"],
            "layer": spec["layer"],
            "expected_dense": spec["expected_dense"],
            **row,
        }
    )

for spec in SNAPSHOT_SPECS:
    filters, params = build_filters(pool=POOL, pool_col="pool", time_col=None)
    sql = dedent(
        f"""
        SELECT count() AS row_count
        FROM {spec['table']}
        {filters}
        """
    )
    stats = query_df(sql, params)
    row = stats.iloc[0].to_dict() if not stats.empty else {}
    inventory_rows.append(
        {
            "table": spec["table"],
            "layer": spec["layer"],
            "expected_dense": None,
            **row,
        }
    )

inventory_df = pd.DataFrame(inventory_rows).sort_values(["layer", "table"]).reset_index(drop=True)
inventory_df


1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows
1 rows


,table,layer,expected_dense,row_count,min_date,max_date,distinct_days
0,int_active_liquidity_daily,intermediate,False,1774,2021-05-05,2026-03-13,1774.0
1,int_active_tvl_daily,intermediate,False,1774,2021-05-05,2026-03-13,1774.0
2,int_current_state,intermediate,None,1,NaT,NaT,NaN
3,int_daily_close,intermediate,False,1774,2021-05-05,2026-03-13,1774.0
4,int_lp_events,intermediate,False,408795,2021-05-05,2026-03-13,1773.0
5,int_range_liquidity_daily,intermediate,False,2866101,2021-05-05,2026-03-13,1774.0
6,int_tick_changes,intermediate,False,693151,2021-05-05,2023-11-15,338.0
7,mart_fee_yield_daily,mart,True,1774,2021-05-05,2026-03-13,1774.0
8,mart_open_positions,mart,None,2999,NaT,NaT,NaN
9,mart_tick_daily,mart,False,153822,2021-05-05,2023-11-15,338.0


In [ ]:
series_frames = {}

for spec in TIME_SERIES_SPECS:
    print(f"Loading {spec['table']}...")
    series_frames[spec["table"]] = load_daily_series(spec, pool=POOL, date_from=DATE_FROM, date_to=DATE_TO)

list(series_frames)


Loading stg_swaps...


1,774 rows
Loading stg_mints...
1,767 rows
Loading stg_burns...
1,773 rows
Loading int_lp_events...
1,773 rows
Loading int_daily_close...
1,774 rows
Loading int_tick_changes...
338 rows
Loading int_range_liquidity_daily...
1,774 rows
Loading int_active_liquidity_daily...
1,774 rows
Loading int_active_tvl_daily...
1,774 rows
Loading mart_volume_daily...
338 rows
Loading mart_tvl_daily...
1,774 rows
Loading mart_fee_yield_daily...
1,774 rows
Loading mart_tick_daily...
338 rows


['stg_swaps',
 'stg_mints',
 'stg_burns',
 'int_lp_events',
 'int_daily_close',
 'int_tick_changes',
 'int_range_liquidity_daily',
 'int_active_liquidity_daily',
 'int_active_tvl_daily',
 'mart_volume_daily',
 'mart_tvl_daily',
 'mart_fee_yield_daily',
 'mart_tick_daily']

## Row counts par table

Si une coupure apparait deja ici dans une table amont, l'erreur est probablement en amont ou dans la source.
Si la coupure n'apparait qu'en aval, elle vient plus probablement d'une transformation.

In [ ]:
for spec in TIME_SERIES_SPECS:
    table_name = spec["table"]
    df = series_frames[table_name]
    plot_timeseries(
        df,
        title=f"{table_name} - daily row count ({spec['layer']}) - {POOL}",
        bar_cols=["row_count"],
    )


## Price lineage

Le but ici est de comparer le dernier prix journalier des swaps au `close_price` intermediaire puis au `eth_price_close` du mart final.
Si `stg_swaps_last_price` bouge mais que `int_daily_close_price` ou `mart_tvl_close_price` reste plat, le bug est dans une transformation.


In [ ]:
swap_filters, swap_params = build_filters(
    pool=POOL,
    date_from=DATE_FROM,
    date_to=DATE_TO,
    pool_col="pool",
    time_col="block_time",
    time_type="datetime",
)

swap_price_df = query_df(
    dedent(
        f"""
        SELECT
            toDate(block_time) AS date,
            argMax(price_token1_in_token0, tuple(block_number, log_index)) AS stg_swaps_last_price,
            avg(price_token1_in_token0) AS stg_swaps_avg_price,
            min(price_token1_in_token0) AS stg_swaps_min_price,
            max(price_token1_in_token0) AS stg_swaps_max_price
        FROM stg_swaps
        {swap_filters}
        GROUP BY date
        ORDER BY date
        """
    ),
    swap_params,
)

daily_close_filters, daily_close_params = build_filters(pool=POOL, date_from=DATE_FROM, date_to=DATE_TO)

daily_close_df = query_df(
    dedent(
        f"""
        SELECT date, close_price AS int_daily_close_price, close_tick
        FROM int_daily_close
        {daily_close_filters}
        ORDER BY date
        """
    ),
    daily_close_params,
)

active_liq_price_df = query_df(
    dedent(
        f"""
        SELECT date, close_price AS int_active_liquidity_close_price
        FROM int_active_liquidity_daily
        {daily_close_filters}
        ORDER BY date
        """
    ),
    daily_close_params,
)

mart_tvl_price_df = query_df(
    dedent(
        f"""
        SELECT date, price_token1_in_token0 AS mart_tvl_close_price
        FROM mart_tvl_daily
        {daily_close_filters}
        ORDER BY date
        """
    ),
    daily_close_params,
)

price_lineage_df = swap_price_df.merge(daily_close_df, on="date", how="outer")
price_lineage_df = price_lineage_df.merge(active_liq_price_df, on="date", how="outer")
price_lineage_df = price_lineage_df.merge(mart_tvl_price_df, on="date", how="outer")
price_lineage_df = price_lineage_df.sort_values("date")

fig = go.Figure()
for column in [
    "stg_swaps_last_price",
    "int_daily_close_price",
    "int_active_liquidity_close_price",
    "mart_tvl_close_price",
]:
    if column in price_lineage_df.columns:
        fig.add_trace(go.Scatter(x=price_lineage_df["date"], y=price_lineage_df[column], mode="lines+markers", name=column))

fig.add_trace(go.Scatter(x=price_lineage_df["date"], y=price_lineage_df["stg_swaps_min_price"], mode="lines", name="stg_swaps_min_price", line={"dash": "dot"}))
fig.add_trace(go.Scatter(x=price_lineage_df["date"], y=price_lineage_df["stg_swaps_max_price"], mode="lines", name="stg_swaps_max_price", line={"dash": "dot"}))
fig.update_layout(title=f"Price lineage - {POOL}", template="plotly_white", hovermode="x unified")
fig.show()

price_lineage_df.tail(20)


1,774 rows
1,774 rows
1,774 rows
1,774 rows


,date,stg_swaps_last_price,stg_swaps_avg_price,stg_swaps_min_price,stg_swaps_max_price,int_daily_close_price,close_tick,int_active_liquidity_close_price,mart_tvl_close_price
1754,2026-02-22,1958.199742,1960.124836,1924.025372,1981.910078,1958.199742,200522,1958.199742,1958.199742
1755,2026-02-23,1855.311013,1882.222131,1764.361190,1958.045991,1855.311013,201062,1855.311013,1855.311013
1756,2026-02-24,1851.475110,1837.962583,1800.179055,1869.376288,1851.475110,201082,1851.475110,1851.475110
1757,2026-02-25,2056.592102,1991.533994,1847.722766,2136.378191,2056.592102,200032,2056.592102,2056.592102
1758,2026-02-26,2027.650617,2043.117960,1977.371514,2097.899356,2027.650617,200173,2027.650617,2027.650617
1759,2026-02-27,1930.007024,1975.012902,1888.043246,2062.272542,1930.007024,200667,1930.007024,1930.007024
1760,2026-02-28,1964.480182,1905.436230,1812.137756,1981.830744,1964.480182,200490,1964.480182,1964.480182
1761,2026-03-01,1938.945713,1984.104672,1909.162892,2051.859801,1938.945713,200621,1938.945713,1938.945713
1762,2026-03-02,2027.148208,1993.361932,1922.946450,2088.272637,2027.148208,200176,2027.148208,2027.148208
1763,2026-03-03,1982.366082,1981.425107,1930.463638,2040.004389,1982.366082,200399,1982.366082,1982.366082


In [ ]:
constant_rows = []

for model_name, value_col in [
    ("stg_swaps_last_price", "stg_swaps_last_price"),
    ("int_daily_close", "int_daily_close_price"),
    ("int_active_liquidity_daily", "int_active_liquidity_close_price"),
    ("mart_tvl_daily", "mart_tvl_close_price"),
]:
    streaks = constant_streaks(price_lineage_df[["date", value_col]].copy(), value_col=value_col, min_days=CONSTANT_PRICE_MIN_DAYS)
    if not streaks.empty:
        streaks = streaks.assign(model=model_name)
        constant_rows.append(streaks)

constant_price_df = pd.concat(constant_rows, ignore_index=True) if constant_rows else pd.DataFrame(columns=["model", "start_date", "end_date", "days", "value"])
constant_price_df = constant_price_df[["model", "start_date", "end_date", "days", "value"]] if not constant_price_df.empty else constant_price_df
constant_price_df


,model,start_date,end_date,days,value


## Gap summary

Attention: certaines tables sont naturellement clairsemees (`stg_mints`, `stg_burns`, `int_lp_events`, `mart_volume_daily`).
Les tables marquees `expected_dense=True` sont celles ou les trous sont les plus suspects.

In [ ]:
gap_rows = []

for spec in TIME_SERIES_SPECS:
    df = series_frames[spec["table"]]
    missing_days, longest_gap_days = gap_summary(df)
    gap_rows.append(
        {
            "table": spec["table"],
            "layer": spec["layer"],
            "expected_dense": spec["expected_dense"],
            "row_count": int(df["row_count"].sum()) if (not df.empty and "row_count" in df.columns) else 0,
            "min_date": df["date"].min() if not df.empty else pd.NaT,
            "max_date": df["date"].max() if not df.empty else pd.NaT,
            "missing_days": missing_days,
            "longest_gap_days": longest_gap_days,
        }
    )

gap_df = pd.DataFrame(gap_rows).sort_values(["expected_dense", "missing_days", "longest_gap_days", "table"], ascending=[False, False, False, True])
gap_df


,table,layer,expected_dense,row_count,min_date,max_date,missing_days,longest_gap_days
11,mart_fee_yield_daily,mart,True,1774,2021-05-05,2026-03-13,0,0
10,mart_tvl_daily,mart,True,1774,2021-05-05,2026-03-13,0,0
5,int_tick_changes,intermediate,False,693151,2021-05-05,2023-11-15,587,386
12,mart_tick_daily,mart,False,153822,2021-05-05,2023-11-15,587,386
9,mart_volume_daily,mart,False,338,2021-05-05,2023-11-15,587,386
1,stg_mints,staging,False,195160,2021-05-05,2026-03-13,7,2
3,int_lp_events,intermediate,False,408795,2021-05-05,2026-03-13,1,1
2,stg_burns,staging,False,213635,2021-05-05,2026-03-13,1,1
7,int_active_liquidity_daily,intermediate,False,1774,2021-05-05,2026-03-13,0,0
8,int_active_tvl_daily,intermediate,False,1774,2021-05-05,2026-03-13,0,0


## Metriques journaliere par table

Ici on trace, table par table, les metriques principales pour voir ou la forme de la serie change.

In [ ]:
for spec in TIME_SERIES_SPECS:
    table_name = spec["table"]
    df = series_frames[table_name]
    metric_cols = [column for column in df.columns if column != "date"]
    line_cols = [column for column in metric_cols if column != "row_count"]
    plot_timeseries(
        df,
        title=f"{table_name} - daily metrics ({spec['layer']}) - {POOL}",
        bar_cols=["row_count"] if "row_count" in df.columns else [],
        line_cols=line_cols,
    )
    display(df.tail(10))


,date,row_count,abs_amount0_usdc,last_price_usdc
1764,2026-03-04,10083,4.591903e+08,2128.411394
1765,2026-03-05,7135,2.742266e+08,2072.581642
1766,2026-03-06,6513,2.362540e+08,1978.642945
1767,2026-03-07,4615,8.437630e+07,1969.920300
1768,2026-03-08,5970,1.922067e+08,1936.706376
1769,2026-03-09,7619,3.225357e+08,1994.127805
1770,2026-03-10,6543,2.586675e+08,2036.422091
1771,2026-03-11,5433,2.057335e+08,2051.511981
1772,2026-03-12,6399,2.780891e+08,2073.405989
1773,2026-03-13,5107,1.857386e+08,2163.304930


,date,row_count,liquidity_delta,abs_amount0_usdc
1757,2026-03-04,224,5.129407e+20,1.217611e+07
1758,2026-03-05,207,4.647449e+20,5.163820e+06
1759,2026-03-06,175,4.058965e+20,5.915542e+06
1760,2026-03-07,164,1.993137e+21,2.623007e+07
1761,2026-03-08,157,1.559108e+21,1.665676e+07
1762,2026-03-09,171,9.160214e+20,1.018684e+07
1763,2026-03-10,160,6.771119e+20,6.363717e+06
1764,2026-03-11,177,1.871195e+21,2.194164e+07
1765,2026-03-12,161,1.762809e+21,1.994119e+07
1766,2026-03-13,114,7.578482e+20,5.345338e+06


,date,row_count,liquidity_delta,abs_amount0_usdc
1763,2026-03-04,278,5.126304e+20,1.251096e+07
1764,2026-03-05,206,4.649692e+20,4.961020e+06
1765,2026-03-06,202,4.059134e+20,5.781469e+06
1766,2026-03-07,151,1.992671e+21,2.643621e+07
1767,2026-03-08,170,1.558916e+21,1.647439e+07
1768,2026-03-09,181,9.161186e+20,1.022330e+07
1769,2026-03-10,220,6.772428e+20,6.384069e+06
1770,2026-03-11,287,1.870899e+21,2.165915e+07
1771,2026-03-12,165,1.762801e+21,1.957024e+07
1772,2026-03-13,211,7.580159e+20,5.685261e+06


,date,row_count,net_liquidity_delta,abs_amount0_usdc
1763,2026-03-04,502,3.102954e+17,2.468707e+07
1764,2026-03-05,413,-2.243418e+17,1.012484e+07
1765,2026-03-06,377,-1.688202e+16,1.169701e+07
1766,2026-03-07,315,4.659729e+17,5.266628e+07
1767,2026-03-08,327,1.922636e+17,3.313115e+07
1768,2026-03-09,352,-9.714702e+16,2.041014e+07
1769,2026-03-10,380,-1.309183e+17,1.274779e+07
1770,2026-03-11,464,2.959876e+17,4.360079e+07
1771,2026-03-12,326,8.842983e+15,3.951144e+07
1772,2026-03-13,325,-1.676965e+17,1.103060e+07


,date,row_count,close_tick,close_price
1764,2026-03-04,1,199688,2128.411394
1765,2026-03-05,1,199954,2072.581642
1766,2026-03-06,1,200418,1978.642945
1767,2026-03-07,1,200462,1969.920300
1768,2026-03-08,1,200632,1936.706376
1769,2026-03-09,1,200340,1994.127805
1770,2026-03-10,1,200130,2036.422091
1771,2026-03-11,1,200056,2051.511981
1772,2026-03-12,1,199950,2073.405989
1773,2026-03-13,1,199526,2163.304930


,date,row_count,volume_usdc,swap_count
328,2023-11-06,1538,2.906269e+08,4389
329,2023-11-07,1613,2.570399e+08,4671
330,2023-11-08,1123,1.727678e+08,3764
331,2023-11-09,3230,7.537988e+08,7134
332,2023-11-10,2297,4.121068e+08,5901
333,2023-11-11,1860,2.890797e+08,4940
334,2023-11-12,1335,1.806146e+08,4012
335,2023-11-13,2258,3.979578e+08,4672
336,2023-11-14,2168,4.139970e+08,4572
337,2023-11-15,616,1.139315e+08,1542


... (3 outputs left out)

,date,row_count,volume_usdc,fees_usdc,swap_count
328,2023-11-06,1,2.906269e+08,145313.466335,4389
329,2023-11-07,1,2.570299e+08,128514.966345,4666
330,2023-11-08,1,1.727772e+08,86388.616158,3768
331,2023-11-09,1,7.537964e+08,376898.199162,7133
332,2023-11-10,1,4.121029e+08,206051.433305,5898
333,2023-11-11,1,2.890750e+08,144537.521480,4943
334,2023-11-12,1,1.806225e+08,90311.226688,4010
335,2023-11-13,1,3.979478e+08,198973.914929,4674
336,2023-11-14,1,4.139797e+08,206989.854510,4567
337,2023-11-15,1,1.139625e+08,56981.259089,1549


,date,row_count,tvl_usd,eth_price_close
1764,2026-03-04,1,8.702796e+07,2128.411394
1765,2026-03-05,1,8.627639e+07,2072.581642
1766,2026-03-06,1,8.440399e+07,1978.642945
1767,2026-03-07,1,8.344570e+07,1969.920300
1768,2026-03-08,1,8.276868e+07,1936.706376
1769,2026-03-09,1,8.427650e+07,1994.127805
1770,2026-03-10,1,8.539681e+07,2036.422091
1771,2026-03-11,1,8.588936e+07,2051.511981
1772,2026-03-12,1,8.650679e+07,2073.405989
1773,2026-03-13,1,8.746001e+07,2163.304930


,date,row_count,tvl_total_usd,tvl_active_usd,fees_usdc
1764,2026-03-04,1,8.702796e+07,6.761427e+07,0.0
1765,2026-03-05,1,8.627639e+07,6.759764e+07,0.0
1766,2026-03-06,1,8.440399e+07,7.395685e+07,0.0
1767,2026-03-07,1,8.344570e+07,7.256608e+07,0.0
1768,2026-03-08,1,8.276868e+07,7.162618e+07,0.0
1769,2026-03-09,1,8.427650e+07,7.536752e+07,0.0
1770,2026-03-10,1,8.539681e+07,6.664556e+07,0.0
1771,2026-03-11,1,8.588936e+07,6.715768e+07,0.0
1772,2026-03-12,1,8.650679e+07,6.785349e+07,0.0
1773,2026-03-13,1,8.746001e+07,6.723836e+07,0.0


,date,row_count,time_seconds,volume_usdc,swap_count
328,2023-11-06,223,86412,2.906269e+08,4389
329,2023-11-07,262,86472,2.570399e+08,4671
330,2023-11-08,151,86328,1.727678e+08,3764
331,2023-11-09,892,86400,7.537988e+08,7134
332,2023-11-10,306,86496,4.121068e+08,5901
333,2023-11-11,258,86328,2.890797e+08,4940
334,2023-11-12,196,86460,1.806146e+08,4012
335,2023-11-13,384,86352,3.979578e+08,4672
336,2023-11-14,452,86496,4.139970e+08,4572
337,2023-11-15,136,37788,1.136560e+08,1536


## Snapshot tables

Ces tables ne sont pas journalieres, donc on les inspecte via un preview et une visualisation adaptee.

In [ ]:
stg_pools_df = query_df("SELECT * FROM stg_pools ORDER BY pool")
display(stg_pools_df)
if not stg_pools_df.empty:
    fig = px.bar(stg_pools_df, x="pool", y="tick_spacing", color="token0_symbol", hover_data=["fee_bps", "deployment_ts"], title="stg_pools - pool metadata", template="plotly_white")
    fig.show()

current_state_df = query_df("SELECT * FROM int_current_state ORDER BY pool")
display(current_state_df)
if not current_state_df.empty:
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=current_state_df["pool"], y=current_state_df["current_tick"], name="current_tick"), secondary_y=False)
    fig.add_trace(go.Scatter(x=current_state_df["pool"], y=current_state_df["price_token1_in_token0"], mode="lines+markers", name="price_token1_in_token0"), secondary_y=True)
    fig.update_layout(title="int_current_state - current tick and price", template="plotly_white")
    fig.show()

open_positions_filters, open_positions_params = build_filters(pool=POOL, pool_col="pool", time_col=None)
open_positions_df = query_df(
    dedent(
        f"""
        SELECT *
        FROM mart_open_positions
        {open_positions_filters}
        ORDER BY net_liquidity DESC
        LIMIT {PREVIEW_LIMIT}
        """
    ),
    open_positions_params,
)
display(open_positions_df)
if not open_positions_df.empty:
    # net_liquidity is Int256 -> arrives as Python ints (object dtype); plotly 'size' needs numeric
    open_positions_df["net_liquidity"] = open_positions_df["net_liquidity"].astype(float)
    fig = px.scatter(
        open_positions_df,
        x="tick_lower",
        y="tick_upper",
        size="net_liquidity",
        color="is_active",
        hover_data=["owner", "position_token0"],
        title=f"mart_open_positions - {POOL}",
        template="plotly_white",
    )
    fig.show()

distribution_filters, distribution_params = build_filters(pool=POOL, pool_col="pool", time_col=None)
distribution_df = query_df(
    dedent(
        f"""
        SELECT *
        FROM mart_tvl_distribution
        {distribution_filters}
        ORDER BY tick_lower
        LIMIT {PREVIEW_LIMIT}
        """
    ),
    distribution_params,
)
display(distribution_df)
if not distribution_df.empty:
    fig = px.bar(
        distribution_df,
        x="tick_mid",
        y="tvl_token0",
        color="is_active_bin",
        hover_data=["liquidity", "price_lower", "price_upper"],
        title=f"mart_tvl_distribution - {POOL}",
        template="plotly_white",
    )
    fig.show()


1 rows


,pool,fee_bps,tick_spacing,token0_address,token0_symbol,token0_decimals,token1_address,token1_symbol,token1_decimals,deployment_block,deployment_ts
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,0.05,10,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,USDC,6,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,WETH,18,12376729,2021-05-05 21:42:11


1 rows


,pool,current_tick,sqrt_price_current,eth_price_latest
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,200376,22433.884332,1986.968798


NameError: name 'make_subplots' is not defined

In [ ]:
# Exploration libre.
preview("mart_tvl_daily", limit=10)
